# Module 05: Calibration Basics

**ECBS5200 Pre-Work** — Practical Deep Learning Engineering

In this notebook, we'll build intuition for model calibration using
synthetic data. No GPU needed, no model training — just numpy and
matplotlib. By the end, you'll understand what calibration means,
why most neural networks fail at it, and how to read a reliability diagram.

## Setup

We only need standard scientific Python libraries. Everything runs on CPU
in a few seconds.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.calibration import calibration_curve

np.random.seed(42)

## What is calibration?

When a classifier outputs a probability — say 0.90 — that number should
mean something. Specifically, if you collect all the predictions where the
model said "90% confident," about 90% of those predictions should actually
be correct.

A model whose predicted probabilities match observed frequencies is called
**well-calibrated**. Most neural networks are not.

Let's simulate two models to see the difference.

## Simulating a well-calibrated model

We'll generate 5,000 synthetic predictions. For a well-calibrated model,
the predicted probability directly determines the true outcome. If the model
says 0.75, there's a 75% chance the prediction is correct.

In [ ]:
n_samples = 5000

# Generate predicted probabilities uniformly across (0, 1)
calibrated_probs = np.random.uniform(0.0, 1.0, n_samples)

# For a well-calibrated model, the outcome follows the predicted probability
# If model says 0.8, there's an 80% chance the true label matches
calibrated_outcomes = np.array([
    np.random.binomial(1, p) for p in calibrated_probs
])

print(f"Simulated {n_samples} predictions from a well-calibrated model")
print(f"Mean predicted confidence: {calibrated_probs.mean():.3f}")
print(f"Overall accuracy: {calibrated_outcomes.mean():.3f}")

## Simulating an overconfident model

An overconfident model inflates its probabilities. It says "95% confident"
when the true accuracy for those predictions is only about 70%. We simulate
this by pushing probabilities toward the extremes while keeping the actual
outcomes based on more modest true probabilities.

In [ ]:
# Start with the same true probabilities
true_probs = np.random.uniform(0.0, 1.0, n_samples)

# The overconfident model inflates its reported probabilities
# Push everything toward 0 or 1 using a power transformation
dist_from_half = np.abs(true_probs - 0.5) / 0.5  # 0 at center, 1 at extremes
dist_from_half = np.clip(dist_from_half, 1e-9, 1.0)  # avoid 0^0.4 warning
inflated = 0.5 * dist_from_half ** 0.4
overconfident_probs = np.where(true_probs >= 0.5, 0.5 + inflated, 0.5 - inflated)

# But the actual outcomes still follow the TRUE (unInflated) probabilities
overconfident_outcomes = np.array([
    np.random.binomial(1, p) for p in true_probs
])

print(f"Simulated {n_samples} predictions from an overconfident model")
print(f"Mean predicted confidence: {overconfident_probs.mean():.3f}")
print(f"Mean TRUE probability:     {true_probs.mean():.3f}")
print(f"Overall accuracy:          {overconfident_outcomes.mean():.3f}")

## Building reliability diagrams

A reliability diagram (also called a calibration plot) is the standard
way to visualize calibration. We bin predictions by predicted confidence,
then plot the actual accuracy within each bin. A perfectly calibrated
model falls on the diagonal.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Well-calibrated model ---
fraction_positive_cal, mean_predicted_cal = calibration_curve(
    calibrated_outcomes, calibrated_probs, n_bins=10, strategy='uniform'
)

axes[0].plot([0, 1], [0, 1], 'k--', label='Perfect calibration')
axes[0].plot(mean_predicted_cal, fraction_positive_cal, 's-',
             color='#2196F3', markersize=8, linewidth=2,
             label='Well-calibrated model')
axes[0].set_xlabel('Mean predicted confidence', fontsize=12)
axes[0].set_ylabel('Actual accuracy (fraction positive)', fontsize=12)
axes[0].set_title('Well-Calibrated Model', fontsize=14)
axes[0].legend(fontsize=10)
axes[0].set_xlim(0, 1)
axes[0].set_ylim(0, 1)
axes[0].set_aspect('equal')
axes[0].grid(True, alpha=0.3)

# --- Overconfident model ---
fraction_positive_oc, mean_predicted_oc = calibration_curve(
    overconfident_outcomes, overconfident_probs, n_bins=10, strategy='uniform'
)

axes[1].plot([0, 1], [0, 1], 'k--', label='Perfect calibration')
axes[1].plot(mean_predicted_oc, fraction_positive_oc, 's-',
             color='#f44336', markersize=8, linewidth=2,
             label='Overconfident model')
axes[1].set_xlabel('Mean predicted confidence', fontsize=12)
axes[1].set_ylabel('Actual accuracy (fraction positive)', fontsize=12)
axes[1].set_title('Overconfident Model', fontsize=14)
axes[1].legend(fontsize=10)
axes[1].set_xlim(0, 1)
axes[1].set_ylim(0, 1)
axes[1].set_aspect('equal')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('reliability_diagrams.png', dpi=150, bbox_inches='tight')
plt.show()

print("Left: points stay close to the diagonal — calibration is good.")
print("Right: at high confidence, the curve falls BELOW the diagonal —")
print("       the model claims high confidence but delivers lower accuracy.")

## Looking at the numbers

Let's print the bin-by-bin comparison to make the overconfidence concrete.

In [ ]:
print("=== Overconfident Model: Bin-by-Bin ===\n")
print(f"{'Predicted Conf.':>16}  {'Actual Accuracy':>16}  {'Gap':>8}")
print("-" * 44)
for pred, actual in zip(mean_predicted_oc, fraction_positive_oc):
    gap = pred - actual
    flag = " <-- overconfident" if gap > 0.05 else ""
    print(f"{pred:>16.3f}  {actual:>16.3f}  {gap:>+8.3f}{flag}")

## Confidence histogram

Overconfident models also tend to cluster their predictions at the
extremes — they're always very sure of themselves, one way or the other.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(calibrated_probs, bins=20, color='#2196F3', alpha=0.7, edgecolor='white')
axes[0].set_xlabel('Predicted confidence', fontsize=12)
axes[0].set_ylabel('Count', fontsize=12)
axes[0].set_title('Well-Calibrated: Prediction Distribution', fontsize=13)
axes[0].set_xlim(0, 1)

axes[1].hist(overconfident_probs, bins=20, color='#f44336', alpha=0.7, edgecolor='white')
axes[1].set_xlabel('Predicted confidence', fontsize=12)
axes[1].set_ylabel('Count', fontsize=12)
axes[1].set_title('Overconfident: Prediction Distribution', fontsize=13)
axes[1].set_xlim(0, 1)

plt.tight_layout()
plt.savefig('confidence_histograms.png', dpi=150, bbox_inches='tight')
plt.show()

print("The well-calibrated model spreads predictions uniformly.")
print("The overconfident model piles predictions near 0 and 1.")

## Connection to our course task

Throughout this course, we'll be building a complaint classifier that
categorizes consumer complaints into issue types. When your complaint
classifier says it's **90% sure** this is a billing issue, should you
believe it?

If you set a business rule like "auto-route complaints where confidence
is above 80%," you're implicitly assuming the model is calibrated. If
it's overconfident — and it probably is — that 80% threshold will let
through many incorrect predictions.

We won't fix this in pre-work. In **Week 4**, we'll learn techniques
like temperature scaling that can recalibrate a model's outputs after
training. For now, the takeaway is: **be suspicious of raw model
probabilities**.

## Check yourself

Before moving on, make sure you can answer these questions:

1. **What does it mean for a model to be "well-calibrated"?**
   The predicted probabilities match the observed frequencies of being
   correct. If it says 80%, it's right about 80% of the time.

2. **Are most neural networks well-calibrated out of the box?**
   No. Most are systematically overconfident — they produce inflated
   confidence scores.

3. **What is a reliability diagram?**
   A plot with predicted confidence on the x-axis and actual accuracy on
   the y-axis. Perfect calibration falls on the diagonal.

4. **Why does calibration matter for production systems?**
   Because production systems use probability thresholds for routing,
   escalation, and filtering. If the probabilities are miscalibrated,
   those thresholds don't do what you think they do.

5. **If a model's reliability diagram falls below the diagonal at high
   confidence, what does that tell you?**
   The model is overconfident — it claims high confidence but delivers
   lower accuracy than promised.